# Stepper motor test (28BYJ-48 + ULN2003)

**Wiring**
| ULN2003 board | Connect to |
|---|---|
| IN1 | GPIO 17 (Pi pin 11) |
| IN2 | GPIO 18 (Pi pin 12) |
| IN3 | GPIO 27 (Pi pin 13) |
| IN4 | GPIO 22 (Pi pin 15) |
| + (power) | 5V (Pi pin 2 or 4) |
| - (ground) | GND |

The motor needs 5V, not 3.3V. The white 5-pin plug goes into the socket on the driver board.

Run cell 1 once to set up, then play with cell 2.

## 1. Setup

In [ ]:
import time
import lgpio

STEP_PINS = [17, 18, 27, 22]   # IN1, IN2, IN3, IN4

# Half-step sequence: 8 patterns per cycle. 4096 half-steps = one full turn of the output shaft.
HALF_STEP = [
    (1, 0, 0, 0),
    (1, 1, 0, 0),
    (0, 1, 0, 0),
    (0, 1, 1, 0),
    (0, 0, 1, 0),
    (0, 0, 1, 1),
    (0, 0, 0, 1),
    (1, 0, 0, 1),
]
STEPS_PER_TURN = 4096

# Re-running this cell is safe: closing the old handle releases every pin it claimed.
try:
    lgpio.gpiochip_close(gpio)
except (NameError, lgpio.error):
    pass
gpio = lgpio.gpiochip_open(0)
for pin in STEP_PINS:
    lgpio.gpio_claim_output(gpio, pin, 0)


def release():
    """Switch all coils off so the motor doesn't heat up while idle."""
    for pin in STEP_PINS:
        lgpio.gpio_write(gpio, pin, 0)


def run_motor(steps=STEPS_PER_TURN, delay=0.002, clockwise=True):
    """Move the motor a number of half-steps.

    steps     : how many half-steps (4096 = one full turn)
    delay     : seconds between half-steps. 0.001 is about the fastest it manages; 0.01 is slow.
    clockwise : False to go the other way
    """
    seq = HALF_STEP if clockwise else HALF_STEP[::-1]
    try:
        for i in range(steps):
            for pin, on in zip(STEP_PINS, seq[i % 8]):
                lgpio.gpio_write(gpio, pin, on)
            time.sleep(delay)
    finally:
        release()

print("Motor ready")

## 2. Try it
Edit the numbers and re-run. Press the stop (■) button to interrupt a long move.

In [ ]:
run_motor(steps=STEPS_PER_TURN, delay=0.002, clockwise=True)     # one full turn, medium speed
print("done")

In [ ]:
run_motor(steps=STEPS_PER_TURN // 2, delay=0.001, clockwise=False)   # half a turn back, fast
print("done")

## Troubleshooting
- **Buzzes or twitches but doesn't turn**: the pin order is wrong. Change the order in `STEP_PINS` (try swapping the middle two) and re-run cell 1.
- **Nothing at all, no LEDs on the driver board**: the driver board has no power, or the IN wires aren't on the GPIO pins listed above. The board's four LEDs should flicker while it runs.
- **LEDs flicker but motor is silent**: the motor's white plug isn't seated in the driver board, or the board's + pin is on 3.3V instead of 5V.
- **Stalls at delay=0.001**: use 0.0015 or so. That's about the limit for this motor.